In [ ]:
# ===================== 环境配置 =====================
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from jax import flatten_util
import matplotlib.pyplot as plt
from tqdm import tqdm
from jax import vmap,jit,grad
from collections import Counter  # 你要的 count 工具
from NES_VMC import create_machine,hi_ext,hi,sampler,SingleStateAnsatz,\
    NESTotalAnsatz,ha,E_fcis, \
    nes_vmc_gradient,make_metropolis_hastings_step,mcmc_sampler_multichain,init_sampler_state,\
    compute_local_energy_matrix_batch,NES_loss_energy

tensor_edges = [(0,1),(2,3),(4,5),(6,7)]
total_ansatz = NESTotalAnsatz(n_spin_orbitals=4,n_states=2,hidden_dim=12,rngs=nnx.Rngs(13))
machine, graphdef, params = create_machine(total_ansatz)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


$$\Psi(\mathbf{x}) = \det\begin{vmatrix}
\psi_1(\mathbf{x}_1) & \psi_2(\mathbf{x}_1) & \dots & \psi_K(\mathbf{x}_1) \\
\psi_1(\mathbf{x}_2) & \psi_2(\mathbf{x}_2) & \dots & \psi_K(\mathbf{x}_2) \\
\vdots & \vdots & \ddots & \vdots \\
\psi_1(\mathbf{x}_K) & \psi_2(\mathbf{x}_K) & \dots & \psi_K(\mathbf{x}_K)
\end{vmatrix}$$  
但是在代码上往往要使用对数域来避免数值溢出 因此:
$$\ln{\Psi(\mathbf{x})} = \ln\left[\, \det
\begin{pmatrix}
\psi_1(\mathbf{x}_1) & \dots & \psi_K(\mathbf{x}_1) \\
\vdots & \ddots & \vdots \\
\psi_1(\mathbf{x}_K) & \dots & \psi_K(\mathbf{x}_K)
\end{pmatrix}
\,\right]$$ 


In [2]:
tensor_edges = [(0,1),(2,3),(4,5),(6,7)]
total_ansatz = NESTotalAnsatz(n_spin_orbitals=4,n_states=2,hidden_dim=12,rngs=nnx.Rngs(13))
machine, graphdef, params = create_machine(total_ansatz)

In [3]:

grad_logPsi = jax.grad(machine, argnums=0, holomorphic=True)

# 向量化（批量 walker）
vmap_grad_logPsi = jax.vmap(grad_logPsi, in_axes=(None, 0))

def nes_vmc_gradient(ha: nk.operator.DiscreteOperator, graphdef, params, x_batch):
    """
    ✅ 最终正确版：完全对齐 NetKet + NES-VMC 论文
    公式：∇⟨E⟩ = ⟨ (tr(E_loc) - tr(E_mean)) * ∇logΨ* ⟩
    """
    # 1. 批量局域能量矩阵
    E_L_batch = compute_local_energy_matrix_batch(ha, graphdef, params, x_batch)
    E_L_mean = jnp.mean(E_L_batch, axis=0)

    
    # 2. 计算 tr(E_loc) 和 tr(E_mean) → ✅ 加了 real
    tr_E_loc_batch = jnp.real(jnp.trace(E_L_batch, axis1=1, axis2=2))
    tr_E_mean = jnp.real(jnp.trace(E_L_mean))
    
    # 3. 中心化能量
    tr_centered = tr_E_loc_batch - tr_E_mean

    # 4. 计算 ∇logΨ
    dlogPsi_batch = vmap_grad_logPsi(params, x_batch)

    # 5. 核心加权平均
    def weight_and_mean(grad_component):
        weights = tr_centered.reshape( (-1,) + (1,)*(grad_component.ndim - 1) )
        return jnp.mean(weights * jnp.conj(grad_component), axis=0)

    grad = jax.tree.map(weight_and_mean, dlogPsi_batch)

    loss_mean = jnp.mean(tr_E_loc_batch)
    return grad, loss_mean, E_L_mean

In [4]:
sampler_state = init_sampler_state(hi_ext, 16,seed=42)
samples,sampler_state = mcmc_sampler_multichain(
    n_samples_per_chain=100,
    n_warmup=20,
    sampler_state=sampler_state,
    edges=((0, 1), (2, 3),(4,5),(6,7)),
    machine=machine,
    params=params,
)
samples.shape

(100, 16, 8)

In [5]:
compute_local_energy_matrix_batch(ha,graphdef,params,samples.reshape(-1,2,4))

Array([[[-0.74159388-0.05236645j,  0.3210141 -1.32309939j],
        [ 0.03029888+0.09249994j, -0.54309567+0.05236645j]],

       [[-0.6045566 +0.05782084j, -0.02565725+0.28809528j],
        [ 0.03773811+0.06648801j, -0.29393405+0.00529309j]],

       [[-0.6045566 +0.05782084j, -0.02565725+0.28809528j],
        [ 0.03773811+0.06648801j, -0.29393405+0.00529309j]],

       ...,

       [[-0.6045566 +0.05782084j, -0.02565725+0.28809528j],
        [ 0.03773811+0.06648801j, -0.29393405+0.00529309j]],

       [[-0.74159388-0.05236645j,  0.3210141 -1.32309939j],
        [ 0.03029888+0.09249994j, -0.54309567+0.05236645j]],

       [[-0.18902934-0.19186271j, -0.1437874 -0.45395143j],
        [-0.02248281+0.02248145j, -0.39209594+0.06636209j]]],      dtype=complex128)

In [6]:
import jax
import jax.numpy as jnp
from jax.flatten_util import ravel_pytree

def compute_nes_qgt(machine, graphdef, params, samples, diag_shift=0.01):
    """
    ###########################################################
    ✅ 最终无报错版 | 完全适配 model(x) = (ln detM, lnM[2,2])
    ✅ 严格遵循你的 QGT 公式：<g*g> - <g*><g>
    ✅ 解决 JAX 只能对标量求导的核心限制
    ###########################################################
    """

    # --------------------------
    # 1. 前向：返回 展平后的 lnM (向量，不是矩阵！)
    # --------------------------
    def forward_logM_flat(params, x):
        model = nnx.merge(graphdef, params)
        ln_det_M, ln_M = model(x)        # ln_M: [2,2] 矩阵
        return jnp.ravel(ln_M)            # ✅ 转成向量 [4]，JAX 允许求导

    # --------------------------
    # 2. 单样本梯度（对标量求和后求导，完全合法）
    # --------------------------
    def grad_single(x):
        # 定义：对 展平向量的“实部+虚部”求和 → 变成标量
        def scalar_forward(params, x):
            f = forward_logM_flat(params, x)
            return jnp.real(f).sum() + 1j * jnp.imag(f).sum()

        # 对标量求导 → 不报错！
        grad = jax.grad(scalar_forward, holomorphic=True)(params, x)
        grad_flat, _ = ravel_pytree(grad)
        return grad_flat

    # --------------------------
    # 3. 批量所有样本
    # --------------------------
    grads = jax.vmap(grad_single)(samples)    # [N_samples, N_params]

    # --------------------------
    # 4. 严格按你的公式计算 QGT
    # --------------------------
    term1 = jnp.mean(grads[..., None] * grads[:, None, :].conj(), axis=0)
    g_mean = jnp.mean(grads, axis=0)
    term2 = g_mean[..., None] * g_mean[None, :].conj()
    S = term1 - term2

    # 正则化
    S_reg = S + diag_shift * jnp.eye(S.shape[0], dtype=S.dtype)

    return S_reg, ravel_pytree(params)[1]

In [7]:
compute_nes_qgt(machine,graphdef, params, samples, diag_shift=0.001)[0].shape  # 458,458

(458, 458)

In [8]:
grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                graphdef=graphdef,
                                                params=params,
                                                x_batch=samples.reshape(-1,2,4))
grad_flat , grad_unravel_fn = flatten_util.ravel_pytree(grad)
grad_flat.shape

(458,)

In [ ]:
from NES_VMC import compute_qgt
import time
# ======================
# 超参数
# ======================
N_CHAINS = 16
N_WARMUP = 50
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 20
N_ITER =20

# ======================
# 初始化 ONCE
# ======================
rngs = nnx.Rngs(21)
model = NESTotalAnsatz(4,2,12,rngs=rngs)
machine, graphdef, params = create_machine(model)

optimizer = optax.sgd(learning_rate=0.1)
opt_state = optimizer.init(params)

# ===================== 7. 训练循环（多链版本） =====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (自然梯度下降法)")

print("超参数")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[]
}
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha")
sampler_state = init_sampler_state(hi_ext, N_CHAINS, seed=21)  # 每次迭代换种子避免初始状态固定
start_time = time.time()
for step in range(N_ITER):
    # 1. 生成多链随机初始状态（模仿NetKet，无需手动指定单个initial_state）
    # 2. 多链采样（总样本数=16*63=1008，和原单链一致）
    samples,sampler_state = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        sampler_state=sampler_state,
        edges=((0, 1), (2, 3),(4,5),(6,7)),
        machine=machine,
        params=params,
    )
    #samples = samples.reshape(-1,2,4)

    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 graphdef=graphdef,
                                                 params=params,
                                                 x_batch=samples.reshape(-1,2,4))
    grad = jax.tree_map(lambda x: x*2, grad)
    #model_output = log(\Psi(X)) 
    # qgt_reg,qgt_unravel_fun = compute_nes_qgt(machine, graphdef, params, samples, diag_shift=0.001) 
    # grad_flat , grad_unravel_fn = flatten_util.ravel_pytree(grad)
    
    # # # 自然梯度求解
    # natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    # natural_grad = grad_unravel_fn(natural_grad)
    # grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 记录历史
    if step % 1 == 0 or step == N_ITER - 1:
        total_model =  nnx.merge(graphdef,params)
        log_Psi,log_M  = total_model(samples.reshape(-1,2,4))
        eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
        history['step'].append(step)
        history['E_Lmatrix'].append(E_L_mean)
        history['samples'].append(samples)
        #history['loss'].append(loss_mean)
        #history['natural_grad'].append(natural_grad)
        history['grad_flat'].append(grad_flat)
        history['log_Psi'].append(log_Psi)
        history['log_M'].append(log_M)
        
        # history['energy'].append(loss_mean)
        # history['energy_std'].append(jnp.std(loss_mean))
        # history['error'].append(jnp.abs(loss_mean - E_fcis[0]))
        history['params'].append(params)
        print(f"Step {step:3d} | Loss: {loss_mean}｜eig_vals: {eig_vals}｜E_L_mean: {E_L_mean}")



end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
# print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
# print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
# print(f"绝对误差：{final_error:.6f} Ha")
# print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)


开始多链 NES-VMC 训练 (自然梯度下降法)
超参数
基态能量=-1.01546825 Ha| 第一激发态能量=-0.87542794 Ha


KeyError: 'loss'

In [40]:
history['log_M']

[Array(-0.2820938-0.42875512j, dtype=complex128),
 Array(-0.2807047-0.45681258j, dtype=complex128),
 Array(-0.26548377-0.48012018j, dtype=complex128),
 Array(-0.25427828-0.54905228j, dtype=complex128),
 Array(-0.09770857-0.74144808j, dtype=complex128),
 Array(0.11710313-0.49253098j, dtype=complex128),
 Array(0.08163516-0.4668616j, dtype=complex128),
 Array(0.12836521-0.28388522j, dtype=complex128),
 Array(0.45227144-0.79540793j, dtype=complex128),
 Array(-0.03753394-0.28863226j, dtype=complex128),
 Array(0.04487744-0.49284788j, dtype=complex128),
 Array(0.03461951-0.28864135j, dtype=complex128),
 Array(0.16889389-0.16682497j, dtype=complex128),
 Array(0.26718444-0.29723684j, dtype=complex128),
 Array(0.32514309-0.26625578j, dtype=complex128),
 Array(0.28111882-0.32829149j, dtype=complex128),
 Array(0.26804766-0.34189889j, dtype=complex128),
 Array(0.29338158-0.36726283j, dtype=complex128),
 Array(0.31201762-0.37155989j, dtype=complex128),
 Array(0.32476037-0.36171407j, dtype=complex128

In [ ]:
nes_vmc_gradient(ha,graphdef,history['params'][6],x_batch=history['samples'][6].reshape(-1,2,4))

In [ ]:
test_samples = history['samples'][6].reshape(-1,8)

In [ ]:
test_samples.tolist()

In [ ]:
counter

In [32]:
def summary_sampler(samples:jnp.array):
    samples = samples.reshape(-1,8)
    tuple_test_samples = [tuple(x.tolist()) for x in samples]
    tuple_test_samples = tuple(tuple_test_samples)
    #amples = tuple(samples.tolist())
    counter = Counter(tuple_test_samples)
    return counter

In [33]:
summary_sampler(history['samples'][69])

Counter({(0, 1, 1, 0, 1, 0, 1, 0): 1600,
         (1, 0, 1, 0, 1, 0, 0, 1): 800,
         (1, 0, 0, 1, 1, 0, 1, 0): 400,
         (1, 0, 1, 0, 0, 1, 1, 0): 400})

In [34]:
import jax.numpy as jnp
from collections import Counter
import matplotlib.pyplot as plt
import imageio
import numpy as np
from tqdm import tqdm  # 显示进度条（可选）
plt.rcParams['font.family'] = ['Hiragino Sans GB']
# ---------------------- 你自己的函数（完美版，直接用） ----------------------
def summary_sampler(samples: jnp.array):
    # 重塑成 (N,8)，每行一个8维向量
    samples = samples.reshape(-1, 8)
    # 把每一行转成 tuple（可哈希，Counter可用）
    tuple_samples = [tuple(x.tolist()) for x in samples]
    # 统计频次
    counter = Counter(tuple_samples)
    return counter

# ---------------------- 绘制单张直方图 ----------------------
def plot_histogram(counter, step, save_path="temp.png"):
    # 提取数据
    keys = [str(k) for k in counter.keys()]   # 序列转字符串
    values = list(counter.values())           # 频次
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(keys, values, color='#4287f5', edgecolor='black', alpha=0.8)
    
    # 标注数值
    for bar, v in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(v), ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.title(f"Step {step} - 8维样本分布直方图", fontsize=16)
    plt.xlabel("8维样本 (0/1序列)", fontsize=12)
    plt.ylabel("出现频次", fontsize=12)
    plt.xticks(rotation=30, ha='right')  # 旋转防止重叠
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

# ---------------------- 生成 GIF：从 step 0 ~ 70 动态演化 ----------------------
def make_evolution_gif(history, total_steps=69, gif_name="distribution_evolution.gif"):
    frames = []
    
    print("正在生成每一帧直方图...")
    for step in tqdm(range(65)):  # 0 ~ 70
        # 取出当前步的 samples
        samples = history['samples'][step]
        
        # 统计分布
        counter = summary_sampler(samples)
        
        # 保存临时图片
        plot_histogram(counter, step, "temp_frame.png")
        
        # 读入图片作为帧
        frames.append(imageio.imread("temp_frame.png"))
    
    # 合成GIF
    print("正在合成GIF...")
    imageio.mimsave(gif_name, frames, duration=1.2, loop=0)
    print(f"✅ GIF 已保存：{gif_name}")

# ---------------------- 【直接运行】 ----------------------
if __name__ == "__main__":
    # 运行这句即可！
    make_evolution_gif(history, total_steps=71)

正在生成每一帧直方图...


  0%|          | 0/65 [00:00<?, ?it/s]/var/folders/8x/k_m4pmb11437ktb_r6tjzt2c0000gn/T/ipykernel_10015/4164330026.py:56: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frames.append(imageio.imread("temp_frame.png"))
100%|██████████| 65/65 [00:07<00:00,  8.56it/s]


正在合成GIF...
✅ GIF 已保存：distribution_evolution.gif
